In [1]:
!pip install -q gradio

In [2]:
import requests
import gradio as gr
import pandas as pd
import statsmodels.api as sm
from datetime import datetime, timedelta

In [3]:
import joblib

model = joblib.load("modelo_lightgbm.pkl")
features = joblib.load("features.pkl")

In [4]:
coords = {
    "Ávila": (40.6565, -4.6818),
    "Burgos": (42.3439, -3.6969),
    "León": (42.5987, -5.5671),
    "Salamanca": (40.9701, -5.6635),
    "Segovia": (40.9429, -4.1088),
    "Soria": (41.7636, -2.4649),
    "Valladolid": (41.6523, -4.7245),
    "Zamora": (41.5033, -5.7446)
}

def get_data(coords, provincia):
  # Obtener coordenadas automáticamente
  lat, lon = coords[provincia]

  # URL Open-Meteo
  url = (
      "https://api.open-meteo.com/v1/forecast"
      f"?latitude={lat}"
      f"&longitude={lon}"
      "&hourly="
      "temperature_2m,"
      "relative_humidity_2m,"
      "wind_speed_10m,"
      "precipitation"
      "&forecast_days=2"
  )

  # Petición API
  response = requests.get(url)

  # JSON
  data = response.json()

  # DataFrame
  df_weather = pd.DataFrame({
      "fecha_hora": data["hourly"]["time"],
      "temperatura": data["hourly"]["temperature_2m"],
      "humedad": data["hourly"]["relative_humidity_2m"],
      "viento": data["hourly"]["wind_speed_10m"],
      "precipitacion": data["hourly"]["precipitation"]
  })

  # Convertir fecha
  df_weather["fecha_hora"] = pd.to_datetime(df_weather["fecha_hora"])

  # Obtener mañana
  mañana = (datetime.now() + timedelta(days=1)).date()

  # Filtrar mañana
  df_mañana = df_weather[
      df_weather["fecha_hora"].dt.date == mañana
  ]

  # Resumen meteorológico
  resultado = {
      "provincia": provincia,
      "temperatura_media": round(df_mañana["temperatura"].mean(), 2),
      "temperatura_maxima": df_mañana["temperatura"].max(),
      "temperatura_minima": df_mañana["temperatura"].min(),
      "viento_medio": round(df_mañana["viento"].mean(), 2),
      "viento_maximo": df_mañana["viento"].max(),
      "humedad_media": round(df_mañana["humedad"].mean(), 2),
      "precipitaciones": round(df_mañana["precipitacion"].sum(), 2),
      "fecha_mañana": mañana
  }

  return pd.DataFrame({k: [v] for k, v in resultado.items()})

In [5]:
import gradio as gr
import pandas as pd

mapeo_provincia = {
    "Burgos": 0,
    "León": 1,
    "Salamanca": 2,
    "Segovia": 3,
    "Soria": 4,
    "Valladolid": 5,
    "Zamora": 6,
    "Ávila": 7
}

mapeo_hospital = {
    "C.A.U. Burgos": 0,
    "C.A.U. León": 1,
    "C.A.U. Salamanca": 2,
    "H.U. Río Hortega": 3,
    "H.C.U. Valladolid": 4,
    "H. El Bierzo": 5,
    "H. Santos Reyes": 6,
    "C.A. Zamora": 7,
    "C.A. Segovia": 8,
    "H. Santiago Apóstol": 9,
    "C.A. Ávila": 10,
    "C.A. Soria": 11,
    "H. Medina del Campo": 12
}

hospitales_por_provincia = {
    "Burgos": [
        "C.A.U. Burgos",
        "H. Santos Reyes"
    ],
    "León": [
        "C.A.U. León",
        "H. El Bierzo"
    ],
    "Salamanca": [
        "C.A.U. Salamanca"
    ],
    "Segovia": [
        "C.A. Segovia"
    ],
    "Soria": [
        "C.A. Soria"
    ],
    "Valladolid": [
        "H.U. Río Hortega",
        "H.C.U. Valladolid",
        "H. Medina del Campo"
    ],
    "Zamora": [
        "C.A. Zamora"
    ],
    "Ávila": [
        "C.A. Ávila"
    ]
}

def actualizar_hospitales(provincia):
    hospitales = hospitales_por_provincia[provincia]

    return gr.Dropdown(
        choices=hospitales,
        value=hospitales[0]
    )


def predecir(provincia, hospital, pacientes_ayer, pacientes_22, pacientes_24, pacientes_24_media):

    mañana = datetime.now() + timedelta(days=1)

    try:
        df_aux = get_data(coords, provincia)
        X_futuro = pd.DataFrame({
            "provincia": [mapeo_provincia[provincia]],
            "hospital": [mapeo_hospital[hospital]],
            "mes": [mañana.month],
            "year": [mañana.year],
            "dia_semana": [mañana.weekday()],
            "hora": [23],
            "temperatura_media": df_aux["temperatura_media"],
            "temperatura_maxima": df_aux["temperatura_maxima"],
            "temperatura_minima": df_aux["temperatura_minima"],
            "precipitaciones": df_aux["precipitaciones"],
            "humedad": df_aux["humedad_media"],
            "viento": df_aux["viento_medio"],
            "viento_maximo": df_aux["viento_maximo"],
            "pacientes_ayer": [pacientes_ayer],
            "lag_1": pacientes_22,
            "larg_24": pacientes_24,
            "rolling_24": pacientes_24_media
        })

        prediccion = model.predict(X_futuro)[0]

        return f"👨‍⚕️ Pacientes previstos: {prediccion:.2f}"

    except Exception as e:
        return f"ERROR:\n{str(e)}"

with gr.Blocks(title="Predicción de Urgencias") as demo:

    gr.Markdown(
        """
        # 🏥 Predicción de Pacientes en Urgencias

        Introduce los parámetros y obtén la predicción.
        """
    )

    with gr.Row():

        with gr.Column():

            provincia = gr.Dropdown(
                choices=list(mapeo_provincia.keys()),
                value="Salamanca",
                label="📍 Provincia"
            )

            hospital = gr.Dropdown(
                choices=list(mapeo_hospital.keys()),
                value="C.A.U. Salamanca",
                label="🏥 Hospital"
            )

            pacientes_ayer = gr.Slider(
              minimum=1,
              maximum=1000,
              value=1,
              step=1,
              label="👨‍⚕️ Pacientes ayer"
            )

        with gr.Column():

            pacientes_22 = gr.Slider(
              minimum=1,
              maximum=1000,
              value=1,
              step=1,
              label="👨‍⚕️ Pacientes a las 22:00"
            )

            pacientes_24 = gr.Slider(
                minimum=1,
                maximum=1000,
                value=1,
                step=1,
                label="👨‍⚕️ Pacientes hace 24h"
            )

            pacientes_24_media = gr.Slider(
                minimum=1,
                maximum=1000,
                value=1,
                step=1,
                label="👨‍⚕️ Media pacientes hace 24h"
            )

    provincia.change(
        fn=actualizar_hospitales,
        inputs=provincia,
        outputs=hospital
    )

    boton = gr.Button(
        "🔮 Realizar predicción",
        variant="primary"
    )

    salida = gr.Textbox(
        label="Resultado",
        lines=3
    )

    boton.click(
        fn=predecir,
        inputs=[
            provincia,
            hospital,
            pacientes_ayer,
            pacientes_22,
            pacientes_24,
            pacientes_24_media
        ],
        outputs=salida
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://62bb1ac94d7b25a708.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
